In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")
dbutils.widgets.text("reader_group", "students")
dbutils.widgets.text("reviewer_email", "lbiel@softserve.academy")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")
READER_GROUP = dbutils.widgets.get("reader_group")
REVIEWER_EMAIL = dbutils.widgets.get("reviewer_email")

GOLD_SCHEMA = f"{CATALOG}.{SCHEMA_PREFIX}_gold"

ACCESS_RULES = f"{GOLD_SCHEMA}.access_rules"
FACT_ORDER_ITEMS = f"{GOLD_SCHEMA}.fact_order_items"
AGG_DAILY_SALES = f"{GOLD_SCHEMA}.agg_daily_sales"
DIM_CUSTOMER = f"{GOLD_SCHEMA}.dim_customer"
DIM_COUNTRY = f"{GOLD_SCHEMA}.dim_country"

RLS_FUNCTION = f"{GOLD_SCHEMA}.rls_country_filter"
CLS_FUNCTION = f"{GOLD_SCHEMA}.cls_mask_pii"

COUNTRY_FILTERED_TABLES = [FACT_ORDER_ITEMS, AGG_DAILY_SALES]
PII_COLUMNS = ["customer_name", "email"]
PII_MASK = "***"

READER_TABLES = [
    f"{GOLD_SCHEMA}.dim_date",
    f"{GOLD_SCHEMA}.dim_customer",
    f"{GOLD_SCHEMA}.dim_product",
    DIM_COUNTRY,
    f"{GOLD_SCHEMA}.dim_order_channel",
    FACT_ORDER_ITEMS,
    AGG_DAILY_SALES,
]



#### 1. Access rules 
One row per user and country. `country = NULL` grants all countries. Users without a row see nothing (default deny).

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {ACCESS_RULES} (
    user_email  STRING  NOT NULL COMMENT 'User the rule applies to; compared with current_user()',
    country     STRING           COMMENT 'Order country (ISO-2) the user may see; NULL means all countries',
    can_see_pii BOOLEAN NOT NULL COMMENT 'TRUE if the user may see unmasked customer name and email'
)
COMMENT 'Entitlements for row filters and column masks on gold tables. Not granted to readers.'
""")

In [0]:
spark.sql(f"""
DELETE FROM {ACCESS_RULES}
WHERE user_email IN (current_user(), '{REVIEWER_EMAIL}')
""")

spark.sql(f"""
INSERT INTO {ACCESS_RULES} (user_email, country, can_see_pii)
VALUES
    (current_user(),     NULL, TRUE),
    ('{REVIEWER_EMAIL}', NULL, TRUE)
""")

#### 2. Row-level security
A row is visible when the current user has a rule for its order country or a rule for all countries.
The filter is applied to the fact and to the daily aggregate, so totals cannot bypass it. The fact stores only the country key, so the filter resolves the country code through `dim_country`.

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {RLS_FUNCTION}(order_country STRING)
RETURNS BOOLEAN
COMMENT 'Row filter: TRUE if current_user() may see orders of this country according to access_rules'
RETURN EXISTS (
    SELECT 1
    FROM {ACCESS_RULES} r
    WHERE r.user_email = current_user()
      AND (r.country IS NULL OR r.country = order_country)
)
""")

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {RLS_FUNCTION}(order_country_key BIGINT)
RETURNS BOOLEAN
COMMENT 'Row filter: TRUE if current_user() may see orders of this country according to access_rules'
RETURN EXISTS (
    SELECT 1
    FROM {ACCESS_RULES} r
    JOIN {DIM_COUNTRY} c ON c.country_key = order_country_key
    WHERE r.user_email = current_user()
      AND (r.country IS NULL OR r.country = c.country_code)
)
""")

In [0]:
for table in COUNTRY_FILTERED_TABLES:
    spark.sql(f"ALTER TABLE {table} SET ROW FILTER {RLS_FUNCTION} ON (country_key)")

#### 3. Column-level security
Customer name and email are returned only to users with `can_see_pii = TRUE`; everyone else sees a fixed mask.

In [0]:
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CLS_FUNCTION}(pii_value STRING)
RETURNS STRING
COMMENT 'Column mask: returns the value only if current_user() has can_see_pii = TRUE in access_rules'
RETURN CASE
    WHEN EXISTS (
        SELECT 1
        FROM {ACCESS_RULES} r
        WHERE r.user_email = current_user()
          AND r.can_see_pii
    )
    THEN pii_value
    ELSE '{PII_MASK}'
END
""")

In [0]:
for column in PII_COLUMNS:
    spark.sql(f"ALTER TABLE {DIM_CUSTOMER} ALTER COLUMN {column} SET MASK {CLS_FUNCTION}")

#### 4. Grants
The reader group gets USE SCHEMA on gold and SELECT on business tables. `access_rules` is intentionally not granted.
USE CATALOG is managed by the workspace administrator at catalog level.

In [0]:
spark.sql(f"GRANT USE SCHEMA ON SCHEMA {GOLD_SCHEMA} TO `{READER_GROUP}`")

for table in READER_TABLES:
    spark.sql(f"GRANT SELECT ON TABLE {table} TO `{READER_GROUP}`")

#### 5. Demonstration
Restricts the current user's own rule to a single country and removes PII permission, then restores full access.
Do not run load notebooks while the rule is restricted: they read through the same filters.

In [0]:
DEMO_COUNTRY = "DE"

spark.sql(f"""
UPDATE {ACCESS_RULES}
SET country = '{DEMO_COUNTRY}', can_see_pii = FALSE
WHERE user_email = current_user()
""")

In [0]:
spark.sql(f"""
UPDATE {ACCESS_RULES}
SET country = NULL, can_see_pii = TRUE
WHERE user_email = current_user()
""")